# 02 · StigmaSymp-W: vignette generation & audit
Generates train/val vignettes across **clinical, lay, euphemistic and Hinglish** registers, split *by phrase* so validation measures generalisation to unseen phrasings.

The **test set is human-written** (no LLM): volunteers write vignettes for criteria assigned in advance. See the protocol cell at the end.

In [ ]:
# Colab setup: clone the repo and install training extras
import os
import subprocess
import sys

if "google.colab" in sys.modules and not os.path.exists("RareCare"):
    subprocess.run(["git", "clone", "https://github.com/nush1729/RareCare.git"], check=True)
    os.chdir("RareCare")
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[train]"], check=True)
print("cwd:", os.getcwd())

In [ ]:
!python -m training.stigmasymp --n 6000 --seed 13 --out data/stigmasymp_w

In [ ]:
import pandas as pd

tr = pd.read_json("data/stigmasymp_w/train.jsonl", lines=True)
va = pd.read_json("data/stigmasymp_w/val.jsonl", lines=True)
print(len(tr), len(va))
display(pd.crosstab(tr.register, tr.tier))
tr.sample(8, random_state=1)[["register", "text", "present", "tier"]]

In [ ]:
# Phrase-level leakage audit: no target surface form may appear in both splits
def phrases(df):
    return {r.text[s["start"]:s["end"]].lower() for r in df.itertuples() for s in r.spans}
overlap = phrases(tr) & phrases(va)
print("overlapping phrases:", overlap or "none")
assert not overlap

## Human-written test set protocol
1. Sample criteria uniformly from `rarecare/kg/data/criteria.json` (plus 20% no-criterion controls).
2. Each writer gets a criterion + register (clinical / lay / euphemistic / Hinglish) and writes 3 vignettes **without** seeing `surface_forms.json` or any LLM.
3. A second annotator labels concept spans and tier blind to the assignment; disagreements are adjudicated (report Cohen's κ).
4. Store as `data/human_test.jsonl` (same schema). Never used for training or model selection.